# **Machine Learning Final Project - News Source Classification Pt. 2**
*Tonoya Ahmed, Hanna Pucheu, Angela Tan*

# Part 1: Data Preprocessing


In [ ]:
# imports
import warnings
warnings.filterwarnings("ignore")
!pip install feedparser
!pip install vaderSentiment
!pip install transformers datasets torch scikit-learn

from datasets import Dataset
import torch
import transformers
import pandas as pd
import numpy as np
from google.colab import drive
import feedparser
from urllib.parse import urlparse, unquote
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments

In [ ]:
drive.mount('/content/drive')

## 1a. Cleaning and Preprocessing Functions

Note: data preproccessing looks slightly different for RoBERTa vs other models.

In [ ]:
%%writefile preprocess.py
import pandas as pd
import re
from urllib.parse import urlparse, unquote

def clean_headline(headline):
    if pd.isna(headline):
        return ""
    headline = re.sub(r"\s*\|\s*.*$", "", str(headline))
    headline = re.sub(r"\s+", " ", headline).strip()
    return headline

def headline_from_url(url):
    url = str(url)
    path = urlparse(url).path
    parts = [p for p in path.split("/") if p]

    if not parts:
        return ""

    slug = parts[-1]
    slug = re.sub(r"-?rcna\d+$", "", slug)
    slug = re.sub(r"-?ncna\d+$", "", slug)
    slug = re.sub(r"-?\d+$", "", slug)

    slug = unquote(slug)
    slug = re.sub(r"[-_]+", " ", slug)
    slug = re.sub(r"[^a-zA-Z0-9\s]", " ", slug)
    slug = re.sub(r"\s+", " ", slug).strip()

    return slug

def prepare_data(csv_path: str):
    df = pd.read_csv(csv_path)

    if "headline" in df.columns:
        X = df["headline"].apply(clean_headline).tolist()
    else:
        X = df["url"].apply(headline_from_url).tolist()

    y = df["url"].str.contains("foxnews.com", case=False, na=False).astype(int).tolist()

    return X, y

## 1b. Data

In [ ]:
from preprocess import prepare_data

path = "/content/drive/MyDrive/ML Project/og_headlines_output.csv"
X, y = prepare_data(path)

In [ ]:
news_df = pd.concat([pd.Series(X, name='headline'), pd.Series(y, name='label')], axis=1)

In [ ]:
X_sample,y_sample = prepare_data("/content/drive/MyDrive/ML Project/new_dataset.csv")
sampled_new_data = pd.concat([pd.Series(X_sample, name='headline'),pd.Series(y_sample, name='label')], axis=1)
sampled_new_data = (sampled_new_data.groupby("label", group_keys=False).apply(lambda x: x.sample(n=5500, random_state=42)))
sampled_new_data = sampled_new_data.reset_index(drop=True)

In [ ]:
#join the data
news_df_cleaned = pd.concat([news_df, sampled_new_data])
news_df_cleaned.shape

# Part 2: Modeling

## 2a. Train test split

In [ ]:
X = news_df_cleaned["headline"]
y = news_df_cleaned["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 2b. Hyperparameter tuning

In [ ]:
# HYPER PARAMETER TUNING -> TAKES VERY LONG ~ 32 min
train_df = pd.DataFrame({"headline": X_train, "label": y_train})
test_df = pd.DataFrame({"headline": X_test, "label": y_test})


roberta_train = Dataset.from_pandas(train_df)
roberta_test = Dataset.from_pandas(test_df)

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Model
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels = 2
)

# Tokenize the text
def tokenize_function(examples):
    return tokenizer(
        examples["headline"],
        padding="max_length", # can change to True
        truncation=True,
        max_length=64
    )

roberta_train = roberta_train.map(tokenize_function, batched=True)
roberta_test = roberta_test.map(tokenize_function, batched=True)

# Keep only needed columns
roberta_train = roberta_train.remove_columns(["headline"])
roberta_test = roberta_test.remove_columns(["headline"])

roberta_train.set_format("torch")
roberta_test.set_format("torch")

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

from itertools import product
import pandas as pd
import numpy as np
import torch
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer


search_space = {
    "learning_rate": [1e-5, 2e-5],
    "weight_decay": [0.05, 0.1],
    "label_smoothing_factor": [0.05, 0.1],
    "num_train_epochs": [2],
    "warmup_steps": [25, 50]
}

results = []

keys = list(search_space.keys())
combos = list(product(*search_space.values()))

for i, values in enumerate(combos):
    params = dict(zip(keys, values))
    print(f"\n===== Trial {i+1}/{len(combos)} =====")
    print(params)

    model = RobertaForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./roberta_search/trial_{i}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="no",

        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,

        learning_rate=params["learning_rate"],
        weight_decay=params["weight_decay"],
        label_smoothing_factor=params["label_smoothing_factor"],
        num_train_epochs=params["num_train_epochs"],
        warmup_steps=params["warmup_steps"],

        report_to="none",
        load_best_model_at_end=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=roberta_train,
        eval_dataset=roberta_test,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_result = trainer.evaluate()

    results.append({
        **params,
        "eval_loss": eval_result["eval_loss"],
        "eval_accuracy": eval_result["eval_accuracy"],
        "eval_f1": eval_result["eval_f1"]
    })

    del model
    del trainer
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values("eval_loss")
results_df

The table above shows the optimal hyperparameters that we found through a grid search hyperparameter tuning process. We then manually changed the training arguments for RoBERTa based on the optimal hyperparameters displayed in the tables above to save time.

## 2c. Final RoBERTa Model

In [ ]:
# Create variables to store evaluation metrics

eval_metrics = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

In [ ]:
# RoBERTa Baseline model - RUN ON GPU
!pip install transformers datasets torch scikit-learn

from datasets import Dataset

train_df = pd.DataFrame({"headline": X_train, "label": y_train})
test_df = pd.DataFrame({"headline": X_test, "label": y_test})


roberta_train = Dataset.from_pandas(train_df)
roberta_test = Dataset.from_pandas(test_df)

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Model
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels = 2
)

# Tokenize the text
def tokenize_function(examples):
    return tokenizer(
        examples["headline"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

roberta_train = roberta_train.map(tokenize_function, batched=True)
roberta_test = roberta_test.map(tokenize_function, batched=True)

# Keep only needed columns
roberta_train = roberta_train.remove_columns(["headline"])
roberta_test = roberta_test.remove_columns(["headline"])

roberta_train.set_format("torch")
roberta_test.set_format("torch")

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Training arguments
training_args = TrainingArguments(
    warmup_steps=50,
    output_dir="./roberta_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=roberta_train,
    eval_dataset=roberta_test,
    compute_metrics=compute_metrics,
)
trainer.train()

# Predictions
pred_output = trainer.predict(roberta_test)
y_pred = np.argmax(pred_output.predictions, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

new_row= {
    'Model': 'RoBERTa',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['precision'],
    'Recall': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['recall'],
    'F1-Score': classification_report(y_test, y_pred, output_dict=True)['weighted avg']['f1-score']
}
eval_metrics = pd.concat([eval_metrics, pd.DataFrame([new_row])])



## 2d. Evaluation

In [ ]:
eval_metrics

In [ ]:
# graphs import pandas as pd

log_history = trainer.state.log_history

# keep only eval logs (these happen at end of each epoch)
eval_logs = [log for log in log_history if "eval_loss" in log]

df_logs = pd.DataFrame(eval_logs)

df_logs

In [ ]:
# validation loss per epoch
import matplotlib.pyplot as plt

plt.figure()

plt.plot(df_logs["epoch"], df_logs["eval_loss"], marker="o")

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss per Epoch")

plt.show()

In [ ]:
# accuracy plot
plt.figure()

plt.plot(df_logs["epoch"], df_logs["eval_accuracy"], marker="o")

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy per Epoch")

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["NBC (0)", "Fox (1)"])

disp.plot(cmap="Blues")
plt.title("Confusion Matrix - RoBERTa Model")
plt.show()

# Part 3: Write to model.py + model.pt

In [ ]:
%%writefile model.py
import torch
import numpy as np
from transformers import RobertaTokenizer, RobertaForSequenceClassification

class NewsClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        self.model = RobertaForSequenceClassification.from_pretrained(
            "roberta-base",
            num_labels=2
        )

    def forward(self, batch):
        if isinstance(batch, dict):
            return self.model(**batch).logits

        texts = [str(x) for x in batch]

        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        device = next(self.model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        return self.model(**inputs).logits

    def predict(self, batch):
        self.eval()
        with torch.no_grad():
            logits = self.forward(batch)
            preds = torch.argmax(logits, dim=1)
        return preds.cpu().numpy()

def get_model():
    return NewsClassifier()

In [ ]:
torch.save(model.state_dict(), "model.pt")

In [ ]:
from google.colab import files

files.download("model.py")
files.download("preprocess.py")
files.download("model.pt")